# 00 — Setup do Colab (PathoShift)

Este notebook roda no **runtime do Colab** (uma VM do Google), conectado pela extensão do Colab no VS Code.

**Atenção:** o kernel NÃO roda na sua máquina, então os arquivos locais (`src/`, `configs/`...) **não existem lá**.
Por isso este notebook:
1. monta o Google Drive (para salvar checkpoints e outputs, porque a sessão do Colab cai);
2. clona/atualiza o repositório do GitHub;
3. instala as dependências;
4. confere a GPU e roda `python pathoshift.py describe`;
5. prepara os dados e as features.

Rode as células em ordem, sempre que abrir uma sessão nova.

In [ ]:
# >>> PREENCHA <<<
REPO_URL = "https://github.com/SEU_USUARIO/pathoshift.git"  # se o repo for privado: https://<TOKEN>@github.com/...
BRANCH = "main"

REPO_DIR = "/content/pathoshift"                      # código (disco local da VM, rápido)
DRIVE_DIR = "/content/drive/MyDrive/pathoshift"      # outputs e checkpoints (persistem no Drive)

## 1. Montar o Google Drive
Vai pedir login na conta Google. Se a célula não funcionar pela extensão, use a paleta de comandos do VS Code:
**`Colab: Mount Google Drive to Server`**.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Clonar (1ª vez) ou atualizar (`git pull`) o repositório

In [ ]:
import os

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!git log --oneline -3

## 3. Instalar as dependências
Pulamos `torch` e `ipykernel`: o Colab já tem o PyTorch com CUDA compatível com a GPU dele
(reinstalar baixaria ~2,5 GB) e mexer no `ipykernel` pode quebrar o kernel.

Se o pip atualizar `numpy` ou `pandas`, **reinicie o kernel** (botão Restart) e rode de novo a partir da célula de configuração (REPO_URL).

In [ ]:
!grep -v -E "^(torch|ipykernel)==" requirements.txt > /tmp/requirements_colab.txt
!pip install -q -r /tmp/requirements_colab.txt

## 4. Conferir a GPU (escolha um runtime com GPU T4)

In [ ]:
!nvidia-smi

import torch
print("torch", torch.__version__, "| CUDA disponível:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Sem GPU! Troque o runtime para um com GPU T4."

## 5. Ligar `outputs/` ao Google Drive
Tudo que o código salvar em `outputs/` (checkpoints, predições, métricas) vai direto para o Drive.

In [ ]:
os.makedirs(f"{DRIVE_DIR}/outputs", exist_ok=True)
if not os.path.islink("outputs"):
    !rm -rf outputs
    os.symlink(f"{DRIVE_DIR}/outputs", "outputs")
!ls -la outputs

## 6. Testar o projeto

In [ ]:
!python pathoshift.py describe

## 7. Preparar os dados e as features
O `prepare` baixa o dataset (~650 MB) para o disco local da VM e gera `data/manifest.jsonl` (é determinístico).
As features levam alguns minutos; se já existirem no Drive, são copiadas em vez de recalculadas.

In [ ]:
!python pathoshift.py prepare
!python -m pytest -q

FEAT_DRIVE = f"{DRIVE_DIR}/features.parquet"
if os.path.exists(FEAT_DRIVE):
    !cp {FEAT_DRIVE} data/features.parquet
else:
    !python pathoshift.py features
    !cp data/features.parquet {FEAT_DRIVE}

## 8. Treinar
```
!python pathoshift.py train --model logreg
!python pathoshift.py train --model cnn
```
Ou abra `02_baselines.ipynb` neste mesmo runtime.